## FD_abs

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)

Mounted at /content/drive


In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import sys
import pickle
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import StandardScaler

### Preprocessing the data

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/MCX_data/result_folder/stage2"

# Pick one example file automatically
part_dirs = sorted(
    [d for d in os.listdir(BASE_DIR) if d.startswith("part")],
    key=lambda x: int(x.replace("part", ""))
)

first_part = part_dirs[0]
part_path = os.path.join(BASE_DIR, first_part)

pkl_files = sorted(
    [f for f in os.listdir(part_path) if f.endswith(".pkl")],
    key=lambda x: int(os.path.splitext(x)[0])
)

sample_pkl = os.path.join(part_path, pkl_files[0])

print("Reading:")
print(sample_pkl)

with open(sample_pkl, "rb") as f:
    data = pickle.load(f)

print("\nTop-level object type:")
print(type(data))

Reading:
/content/drive/MyDrive/MCX_data/result_folder/stage2/part1/1.pkl

Top-level object type:
<class 'dict'>


In [ ]:
data.keys()

dict_keys([10, 15, 20, 25, 30, 35, 40, 45])

### Training set

In [ ]:
import os
import re
import csv
import pickle
import numpy as np
from tqdm import tqdm

# ============================================================
# Paths
# ============================================================
BASE_DIR = "/content/drive/MyDrive/MCX_data/result_folder/stage2"
SAVE_CSV = "/content/drive/MyDrive/MCX_data/result_folder/training_fd_110MHz.csv"

# ============================================================
# FD-NIRS conversion settings
# ============================================================
TARGET_FREQ = 110e6   # 110 MHz
TEND_SEC = 1e-8
WAVELENGTHS = ["784", "835"]


# ============================================================
# Your function, slightly cleaned
# returns uac, udc, phase
# ============================================================
def extract_freq(target_freq, TPSF_list, tend):
    """
    Convert one or multiple TD-NIRS TPSFs to FD-NIRS at target_freq.

    Parameters
    ----------
    target_freq : float
        Modulation frequency in Hz, e.g. 100e6.
    TPSF_list : list of array-like
        Each element is one TPSF curve.
    tend : float
        End time of TPSF in seconds.

    Returns
    -------
    amplitude_list : list
        AC amplitude, uac.
    udc_list : list
        DC intensity, integral of TPSF.
    phase_list : list
        Phase in radians.
    """
    amplitude_list = []
    udc_list = []
    phase_list = []
    phase2_list = []

    omega = 2 * np.pi * target_freq

    for TPSF in TPSF_list:
        TPSF = np.asarray(TPSF, dtype=float)
        devf = len(TPSF)

        t = np.linspace(0, tend, devf)

        denominator = np.trapz(TPSF, t)

        if denominator == 0 or np.isnan(denominator):
            amplitude = np.nan
            udc = np.nan
            phase = np.nan
            phase2 = np.nan
        else:
            tau = np.trapz(t * TPSF, t) / denominator

            I_f = np.trapz(TPSF * np.exp(-1j * omega * t), t)

            amplitude = np.abs(I_f)
            phase = np.angle(I_f, deg=False)

            udc = denominator

            # Alternative phase from mean time of flight
            phase2 = -2 * np.pi * target_freq * tau

            # Phase correction from your original code
            if phase > 0 and phase2 < 0:
                phase = phase - 2 * np.pi

        amplitude_list.append(amplitude)
        udc_list.append(udc)
        phase_list.append(phase)
        phase2_list.append(phase2)

    return amplitude_list, udc_list, phase_list, phase2_list


# ============================================================
# Sorting helper
# ============================================================
def numeric_sort_key(name):
    """
    Sort part1, part2, ..., part20 and 1.pkl, 2.pkl, ...
    """
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 10**12


# ============================================================
# Collect all part folders
# ============================================================
part_dirs = [
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d)) and d.startswith("part")
]

part_dirs = sorted(part_dirs, key=numeric_sort_key)

print("Found part folders:")
print(part_dirs)


# ============================================================
# Write CSV directly, memory-safe
# ============================================================
fieldnames = [
    "part",
    "pkl_file",
    "simulation_id",
    "sds_key",
    "wavelength_index",
    "wavelength",
    "target_freq_hz",
    "uac",
    "udc",
    "phase_rad",
    "phase2_rad",
    "tend_sec",
    "n_time_points"
]

total_files = 0
total_rows = 0

with open(SAVE_CSV, "w", newline="") as f_csv:
    writer = csv.DictWriter(f_csv, fieldnames=fieldnames)
    writer.writeheader()

    for part in part_dirs:
        part_path = os.path.join(BASE_DIR, part)

        pkl_files = [
            f for f in os.listdir(part_path)
            if f.endswith(".pkl")
        ]
        pkl_files = sorted(pkl_files, key=numeric_sort_key)

        print(f"\nProcessing {part}: {len(pkl_files)} pickle files")

        for pkl_file in tqdm(pkl_files):
            pkl_path = os.path.join(part_path, pkl_file)

            with open(pkl_path, "rb") as f:
                data = pickle.load(f)

            # data keys: 10, 15, 20, ..., 45
            for sds_key in sorted(data.keys()):
                TPSF_2wls = data[sds_key]

                # TPSF_2wls should contain 2 elements, one per wavelength
                uac_list, udc_list, phase_list, phase2_list = extract_freq(
                    target_freq=TARGET_FREQ,
                    TPSF_list=TPSF_2wls,
                    tend=TEND_SEC
                )

                for wl_idx in range(len(TPSF_2wls)):
                    wavelength = WAVELENGTHS[wl_idx] if wl_idx < len(WAVELENGTHS) else f"wl{wl_idx+1}"

                    writer.writerow({
                        "part": part,
                        "pkl_file": pkl_file,
                        "simulation_id": os.path.splitext(pkl_file)[0],
                        "sds_key": sds_key,
                        "wavelength_index": wl_idx,
                        "wavelength": wavelength,
                        "target_freq_hz": TARGET_FREQ,
                        "uac": uac_list[wl_idx],
                        "udc": udc_list[wl_idx],
                        "phase_rad": phase_list[wl_idx],
                        "phase2_rad": phase2_list[wl_idx],
                        "tend_sec": TEND_SEC,
                        "n_time_points": len(TPSF_2wls[wl_idx])
                    })

                    total_rows += 1

            total_files += 1

print("\nDone.")
print(f"Total pickle files processed: {total_files}")
print(f"Total CSV rows saved: {total_rows}")
print(f"Saved to: {SAVE_CSV}")

Found part folders:
['part1', 'part2', 'part3', 'part4', 'part5', 'part6', 'part7', 'part8', 'part9', 'part10', 'part11', 'part12', 'part13', 'part14', 'part15', 'part16', 'part17', 'part18', 'part19', 'part20']

Processing part1: 500 pickle files


  0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_3141/1495676860.py:61: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  denominator = np.trapz(TPSF, t)
/tmp/ipykernel_3141/1495676860.py:69: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  tau = np.trapz(t * TPSF, t) / denominator
/tmp/ipykernel_3141/1495676860.py:71: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  I_f = np.trapz(TPSF * np.exp(-1j * omega * t), t)
100%|██████████| 500/500 [00:29<00:00, 16.96it/s]



Processing part2: 500 pickle files


100%|██████████| 500/500 [00:29<00:00, 16.70it/s]



Processing part3: 500 pickle files


...


Processing part19: 500 pickle files


100%|██████████| 500/500 [00:32<00:00, 15.60it/s]



Processing part20: 500 pickle files


100%|██████████| 500/500 [00:29<00:00, 17.06it/s]


Done.
Total pickle files processed: 10000
Total CSV rows saved: 160000
Saved to: /content/drive/MyDrive/MCX_data/result_folder/training_fd_110MHz.csv


### Testset

In [ ]:
import os
import re
import csv
import pickle
import numpy as np
from tqdm import tqdm

# ============================================================
# Paths
# ============================================================
BASE_DIR = "/content/drive/MyDrive/MCX_data/result_folder/testset"
SAVE_CSV = "/content/drive/MyDrive/MCX_data/result_folder/testing_fd_110MHz.csv"

# ============================================================
# FD-NIRS conversion settings
# ============================================================
TARGET_FREQ = 110e6   # 110 MHz
TEND_SEC = 1e-8
WAVELENGTHS = ["784", "835"]


# ============================================================
# Your function, slightly cleaned
# returns uac, udc, phase
# ============================================================
def extract_freq(target_freq, TPSF_list, tend):
    """
    Convert one or multiple TD-NIRS TPSFs to FD-NIRS at target_freq.

    Parameters
    ----------
    target_freq : float
        Modulation frequency in Hz, e.g. 100e6.
    TPSF_list : list of array-like
        Each element is one TPSF curve.
    tend : float
        End time of TPSF in seconds.

    Returns
    -------
    amplitude_list : list
        AC amplitude, uac.
    udc_list : list
        DC intensity, integral of TPSF.
    phase_list : list
        Phase in radians.
    """
    amplitude_list = []
    udc_list = []
    phase_list = []
    phase2_list = []

    omega = 2 * np.pi * target_freq

    for TPSF in TPSF_list:
        TPSF = np.asarray(TPSF, dtype=float)
        devf = len(TPSF)

        t = np.linspace(0, tend, devf)

        denominator = np.trapz(TPSF, t)

        if denominator == 0 or np.isnan(denominator):
            amplitude = np.nan
            udc = np.nan
            phase = np.nan
            phase2 = np.nan
        else:
            tau = np.trapz(t * TPSF, t) / denominator

            I_f = np.trapz(TPSF * np.exp(-1j * omega * t), t)

            amplitude = np.abs(I_f)
            phase = np.angle(I_f, deg=False)

            udc = denominator

            # Alternative phase from mean time of flight
            phase2 = -2 * np.pi * target_freq * tau

            # Phase correction from your original code
            if phase > 0 and phase2 < 0:
                phase = phase - 2 * np.pi

        amplitude_list.append(amplitude)
        udc_list.append(udc)
        phase_list.append(phase)
        phase2_list.append(phase2)

    return amplitude_list, udc_list, phase_list, phase2_list


# ============================================================
# Sorting helper
# ============================================================
def numeric_sort_key(name):
    """
    Sort part1, part2, ..., part20 and 1.pkl, 2.pkl, ...
    """
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 10**12


# ============================================================
# Collect all part folders
# ============================================================
part_dirs = [
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d)) and d.startswith("part")
]

part_dirs = sorted(part_dirs, key=numeric_sort_key)

print("Found part folders:")
print(part_dirs)


# ============================================================
# Write CSV directly, memory-safe
# ============================================================
fieldnames = [
    "part",
    "pkl_file",
    "simulation_id",
    "sds_key",
    "wavelength_index",
    "wavelength",
    "target_freq_hz",
    "uac",
    "udc",
    "phase_rad",
    "phase2_rad",
    "tend_sec",
    "n_time_points"
]

total_files = 0
total_rows = 0

with open(SAVE_CSV, "w", newline="") as f_csv:
    writer = csv.DictWriter(f_csv, fieldnames=fieldnames)
    writer.writeheader()

    for part in part_dirs:
        part_path = os.path.join(BASE_DIR, part)

        pkl_files = [
            f for f in os.listdir(part_path)
            if f.endswith(".pkl")
        ]
        pkl_files = sorted(pkl_files, key=numeric_sort_key)

        print(f"\nProcessing {part}: {len(pkl_files)} pickle files")

        for pkl_file in tqdm(pkl_files):
            pkl_path = os.path.join(part_path, pkl_file)

            with open(pkl_path, "rb") as f:
                data = pickle.load(f)

            # data keys: 10, 15, 20, ..., 45
            for sds_key in sorted(data.keys()):
                TPSF_2wls = data[sds_key]

                # TPSF_2wls should contain 2 elements, one per wavelength
                uac_list, udc_list, phase_list, phase2_list = extract_freq(
                    target_freq=TARGET_FREQ,
                    TPSF_list=TPSF_2wls,
                    tend=TEND_SEC
                )

                for wl_idx in range(len(TPSF_2wls)):
                    wavelength = WAVELENGTHS[wl_idx] if wl_idx < len(WAVELENGTHS) else f"wl{wl_idx+1}"

                    writer.writerow({
                        "part": part,
                        "pkl_file": pkl_file,
                        "simulation_id": os.path.splitext(pkl_file)[0],
                        "sds_key": sds_key,
                        "wavelength_index": wl_idx,
                        "wavelength": wavelength,
                        "target_freq_hz": TARGET_FREQ,
                        "uac": uac_list[wl_idx],
                        "udc": udc_list[wl_idx],
                        "phase_rad": phase_list[wl_idx],
                        "phase2_rad": phase2_list[wl_idx],
                        "tend_sec": TEND_SEC,
                        "n_time_points": len(TPSF_2wls[wl_idx])
                    })

                    total_rows += 1

            total_files += 1

print("\nDone.")
print(f"Total pickle files processed: {total_files}")
print(f"Total CSV rows saved: {total_rows}")
print(f"Saved to: {SAVE_CSV}")

Found part folders:
['part1', 'part2', 'part3', 'part4', 'part5', 'part6', 'part7', 'part8', 'part9', 'part10']

Processing part1: 100 pickle files


  0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_3141/3662150580.py:61: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  denominator = np.trapz(TPSF, t)
/tmp/ipykernel_3141/3662150580.py:69: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  tau = np.trapz(t * TPSF, t) / denominator
/tmp/ipykernel_3141/3662150580.py:71: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  I_f = np.trapz(TPSF * np.exp(-1j * omega * t), t)
100%|██████████| 100/100 [00:57<00:00,  1.75it/s]



Processing part2: 100 pickle files


100%|██████████| 100/100 [01:06<00:00,  1.51it/s]



Processing part3: 100 pickle files


...


Processing part9: 100 pickle files


100%|██████████| 100/100 [00:57<00:00,  1.73it/s]



Processing part10: 100 pickle files


100%|██████████| 100/100 [00:53<00:00,  1.86it/s]


Done.
Total pickle files processed: 1000
Total CSV rows saved: 16000
Saved to: /content/drive/MyDrive/MCX_data/result_folder/testing_fd_110MHz.csv
